In [ ]:
# ============================
# PHASE 1: Data Collection & Preparation
# ============================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import os
import re
import warnings
warnings.filterwarnings("ignore")

# Create directories
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Directories created!")

In [ ]:
# Download TSLA price data
ticker = "TSLA"
start_date = "2021-01-01"
end_date = "2023-12-31"

print(f"Downloading {ticker} data from {start_date} to {end_date}...")
price_df = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Flatten MultiIndex columns if present (common in newer yfinance versions)
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.droplevel(1)

price_df = price_df[['Open', 'High', 'Low', 'Close', 'Volume']]
price_df.reset_index(inplace=True)

print(f"Downloaded {len(price_df)} days of price data.")
price_df.to_csv("data/raw/tsla_price_raw.csv", index=False)
print("Price data saved to data/raw/tsla_price_raw.csv")
price_df.head()

In [13]:
# Load the main stock tweets dataset
stock_tweets_path = "data/raw/stock_tweets.csv"
print(f"Reading {stock_tweets_path}...")
tweets_df = pd.read_csv(stock_tweets_path)

# Filter for TSLA stock name
tsla_tweets = tweets_df[tweets_df['Stock Name'] == 'TSLA']

# Parse dates and filter to target date range (2021-09-01 to 2022-10-01)
tsla_tweets['Date'] = pd.to_datetime(tsla_tweets['Date']).dt.date
start_target = pd.to_datetime("2021-09-01").date()
end_target = pd.to_datetime("2022-10-01").date()

filtered_tsla_tweets = tsla_tweets[(tsla_tweets['Date'] >= start_target) & (tsla_tweets['Date'] <= end_target)]

# Select and rename the required columns to match schema
filtered_tsla_tweets = filtered_tsla_tweets[['Date', 'Tweet']]

# Save to raw tweets folder
filtered_tsla_tweets.to_csv("data/raw/tsla_tweets_raw.csv", index=False)

print(f"Successfully extracted {len(filtered_tsla_tweets)} TSLA tweets between {start_target} and {end_target}.")
print("TSLA tweets saved to data/raw/tsla_tweets_raw.csv")
filtered_tsla_tweets.head()

Reading data/raw/stock_tweets.csv...
Successfully extracted 37422 TSLA tweets between 2021-09-01 and 2022-10-01.
TSLA tweets saved to data/raw/tsla_tweets_raw.csv


,Date,Tweet
0,2022-09-29,Mainstream media has done an amazing job at br...
1,2022-09-29,Tesla delivery estimates are at around 364k fr...
2,2022-09-29,3/ Even if I include 63.0M unvested RSUs as of...
3,2022-09-29,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...
4,2022-09-29,"@RealDanODowd @Tesla Stop trying to kill kids,..."


In [22]:
import re
import pandas as pd

# 1. Load and clean price data
print("Cleaning price data...")
price_df = pd.read_csv("data/raw/tsla_price_raw.csv")

# Drop any rows where Date is null/corrupt
price_df = price_df.dropna(subset=['Date'])
price_df = price_df[price_df['Date'].astype(str).str.strip() != '']
price_df = price_df[~price_df['Open'].astype(str).str.contains('TSLA', na=False)]

price_df['Date'] = pd.to_datetime(price_df['Date'])
price_df = price_df.sort_values('Date').reset_index(drop=True)
price_df = price_df.ffill()

# 2. Load and clean tweet data
print("Cleaning tweet data...")
tweets = pd.read_csv("data/raw/tsla_tweets_raw.csv")
tweets['Date'] = pd.to_datetime(tweets['Date'])

def clean_tweet(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # remove URLs
    # Removed mention stripping so @tesla is preserved as 'tesla'
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)           # remove special chars and emojis
    text = re.sub(r'\s+', ' ', text).strip()             # remove extra spaces
    return text

tweets['Tweet'] = tweets['Tweet'].apply(clean_tweet)
tweets = tweets[tweets['Tweet'] != ""]  # remove empty
tweets = tweets.drop_duplicates(subset=['Tweet'])

# 3. Merge based on Date
print("Merging datasets...")
merged_df = pd.merge(tweets, price_df, on='Date', how='inner')

# 4. Save processed datasets
os.makedirs("data/processed", exist_ok=True)
price_df.to_csv("data/processed/tsla_price_clean.csv", index=False)
tweets.to_csv("data/processed/tsla_tweets_clean.csv", index=False)
merged_df.to_csv("data/processed/tsla_merged.csv", index=False)

print(f"✅ Price data cleaned: {len(price_df)} rows")
print(f"✅ Tweets cleaned: {len(tweets)} rows")
print(f"✅ Merged dataset created: {len(merged_df)} rows")
merged_df.head()

Cleaning price data...
Cleaning tweet data...
Merging datasets...
✅ Price data cleaned: 753 rows
✅ Tweets cleaned: 36985 rows
✅ Merged dataset created: 29675 rows


,Date,Tweet,Open,High,Low,Close,Volume
0,2022-09-29,mainstream media has done an amazing job at br...,282.76001,283.649994,265.779999,268.209991,77620600
1,2022-09-29,tesla delivery estimates are at around 364k fr...,282.76001,283.649994,265.779999,268.209991,77620600
2,2022-09-29,3 even if i include 630m unvested rsus as of 6...,282.76001,283.649994,265.779999,268.209991,77620600
3,2022-09-29,realdanodowd wholemarsblog tesla hahaha why ar...,282.76001,283.649994,265.779999,268.209991,77620600
4,2022-09-29,realdanodowd tesla stop trying to kill kids yo...,282.76001,283.649994,265.779999,268.209991,77620600
